In [1]:
# Cell 1 — imports
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Cell 2 — config
DATA_PATH  = "final_data.parquet"
MODEL_DIR  = "nilm_models"
WINDOW_SIZE = 11
BATCH_SIZE  = 256
EPOCHS      = 100
LR          = 1e-3
PATIENCE    = 10

APPLIANCE_COLS = [
    "elec_ceiling_fan_kwh",
    "elec_clothes_washer_kwh",
    "elec_cooling_kwh",
    "elec_freezer_kwh",
    "elec_heating_kwh",
    "elec_hot_water_kwh",
    "elec_lighting_exterior_kwh",
    "elec_lighting_interior_kwh",
    "elec_plug_loads_kwh",
    "elec_range_oven_kwh",
    "elec_refrigerator_kwh",
    "elec_television_kwh",
]

os.makedirs(MODEL_DIR, exist_ok=True)

In [3]:
# Cell 3 — replace the model definition cell, add weighted loss

class Seq2Point(nn.Module):
    def __init__(self, window_size=11):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(window_size, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)


class WeightedMSELoss(nn.Module):
    """
    Penalises missing active hours more than false positives.
    active_weight: how much more to penalise active hours vs zero hours.
    """
    def __init__(self, active_weight=10.0):
        super().__init__()
        self.active_weight = active_weight

    def forward(self, pred, target):
        weights = torch.where(
            target > 0,
            torch.full_like(target, self.active_weight),
            torch.ones_like(target)
        )
        return (weights * (pred - target) ** 2).mean()

In [4]:
# Cell 4 — dataset
class NILMDataset(Dataset):
    def __init__(self, aggregate, appliance, window_size=11):
        half = window_size // 2
        X, y = [], []
        padded = np.pad(aggregate, (half, half), mode="edge")
        for i in range(len(aggregate)):
            X.append(padded[i : i + window_size])
            y.append(appliance[i])
        self.X = torch.FloatTensor(np.array(X))
        self.y = torch.FloatTensor(np.array(y)).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [7]:
# Cell 5 — replace train_one_appliance to accept loss_fn as argument

def train_one_appliance(aggregate_train, appliance_train,
                        aggregate_val,   appliance_val,
                        loss_fn,         window_size=WINDOW_SIZE):

    train_ds = NILMDataset(aggregate_train, appliance_train, window_size)
    val_ds   = NILMDataset(aggregate_val,   appliance_val,   window_size)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE)

    model     = Seq2Point(window_size)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=3, factor=0.5
    )

    best_val_loss = float("inf")
    patience_ctr  = 0
    best_state    = None

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for xb, yb in train_dl:
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(xb)
        train_loss /= len(train_ds)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for xb, yb in val_dl:
                val_loss += loss_fn(model(xb), yb).item() * len(xb)
        val_loss /= len(val_ds)

        scheduler.step(val_loss)

        if epoch % 10 == 0:
            print(f"  Epoch {epoch:3d} | train {train_loss:.6f} | val {val_loss:.6f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_ctr  = 0
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"  Early stop at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    return model, best_val_loss

In [8]:
# Cell 6 — replace the training loop

print("Loading data...")
df = pd.read_parquet(DATA_PATH)

aggregate      = df[APPLIANCE_COLS].sum(axis=1).values.astype(np.float32)
scaler         = StandardScaler()
aggregate_norm = scaler.fit_transform(aggregate.reshape(-1, 1)).flatten()
joblib.dump(scaler, os.path.join(MODEL_DIR, "nilm_scaler.pkl"))
print(f"Scaler saved. Mean={scaler.mean_[0]:.4f}, Std={scaler.scale_[0]:.4f}")

split     = int(len(aggregate) * 0.85)
agg_train = aggregate_norm[:split]
agg_val   = aggregate_norm[split:]

# Appliances with tiny kWh values need higher active_weight
HIGH_WEIGHT = [
    "elec_ceiling_fan_kwh",
    "elec_lighting_exterior_kwh",
    "elec_lighting_interior_kwh",
    "elec_freezer_kwh",
    "elec_television_kwh",
    "elec_refrigerator_kwh",
]

SPARSE = [
    "elec_clothes_washer_kwh",
    "elec_heating_kwh",
    "elec_hot_water_kwh",
    "elec_range_oven_kwh",
]

results = {}
for col in APPLIANCE_COLS:
    print(f"\nTraining: {col}")

    # Pick loss function based on appliance characteristics
    if col in HIGH_WEIGHT:
        loss_fn = WeightedMSELoss(active_weight=50.0)  # tiny values need strong weight
    elif col in SPARSE:
        loss_fn = WeightedMSELoss(active_weight=20.0)  # sparse but larger values
    else:
        loss_fn = WeightedMSELoss(active_weight=10.0)  # default

    appliance = df[col].values.astype(np.float32)
    model, val_loss = train_one_appliance(
        agg_train, appliance[:split],
        agg_val,   appliance[split:],
        loss_fn
    )

    path = os.path.join(MODEL_DIR, f"nilm_{col}.pt")
    torch.save(model.state_dict(), path)
    results[col] = val_loss
    print(f"  Saved → {path} (val loss: {val_loss:.6f})")

print("\n── Training complete ──")
print(f"{'Appliance':<35} {'Val Loss':>10}")
print("-" * 47)
for col, loss in results.items():
    print(f"{col:<35} {loss:>10.6f}")

Loading data...
Scaler saved. Mean=0.2183, Std=0.2397

Training: elec_ceiling_fan_kwh
  Epoch   0 | train 0.000282 | val 0.000301
  Epoch  10 | train 0.000277 | val 0.000301
  Early stop at epoch 10
  Saved → nilm_models\nilm_elec_ceiling_fan_kwh.pt (val loss: 0.000301)

Training: elec_clothes_washer_kwh
  Epoch   0 | train 0.001291 | val 0.000680
  Epoch  10 | train 0.001290 | val 0.000680
  Early stop at epoch 10
  Saved → nilm_models\nilm_elec_clothes_washer_kwh.pt (val loss: 0.000680)

Training: elec_cooling_kwh
  Epoch   0 | train 0.039901 | val 0.008669
  Epoch  10 | train 0.026836 | val 0.006776
  Early stop at epoch 13
  Saved → nilm_models\nilm_elec_cooling_kwh.pt (val loss: 0.006104)

Training: elec_freezer_kwh
  Epoch   0 | train 0.000793 | val 0.000710
  Epoch  10 | train 0.000548 | val 0.001019
  Early stop at epoch 11
  Saved → nilm_models\nilm_elec_freezer_kwh.pt (val loss: 0.000670)

Training: elec_heating_kwh
  Epoch   0 | train 0.045977 | val 0.003720
  Epoch  10 | tr